In [ ]:
# --- portable setup (added 2026-09-12 when the project moved to GitHub) ---------------------
# All paths below resolve from HME_ROOT, the repository root.  On Colab: mount Drive and point
# HME_ROOT at your clone.  Locally: run jupyter from the repo, or export HME_ROOT=/path/to/repo.
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    HME_ROOT = os.environ.get('HME_ROOT', '/content/drive/MyDrive/hyperbolic-icd10')   # <-- edit
except ImportError:
    HME_ROOT = os.environ.get('HME_ROOT', os.path.abspath(os.path.join(os.getcwd(), '..')))
assert os.path.isdir(os.path.join(HME_ROOT, 'data')), f'HME_ROOT={HME_ROOT!r} is not the repo root'
print('HME_ROOT =', HME_ROOT)


# Position-Dependent Curvature: c(A) in the metric itself

Every prior attempt made kappa a per-node **distance multiplier**, which the loss largely ignores.
This puts position into the **curvature parameter c** of the Poincare ball, so the shape of the
distance function varies by region — genuinely position-dependent geometry.

**Derivation.** A node's angular sector has width ~e^(-A), A = cumulative ancestral log-branching.
Its subtree of S descendants must fit within radius h*delta in that sector. Volume in curvature -c
grows as e^(sqrt(c)*r), so fitting S nodes in a sector of width e^(-A) requires

    sqrt(c) * h * delta  >=  log S + A     ->     c(u) prop. ( (log S_u + A_u) / h_u )^2

Crowded, shallow-subtree regions need high curvature. Sparse, deep regions need less.

**Pair distance** uses c_uv = sqrt(c_u * c_v) (geometric mean — standard in mixed-curvature work,
reduces exactly to c when both endpoints share it).

Run order for every experiment: **d = 5, 10, 2**, and **position-dependent always trains before constant**.

## 1. Setup

In [3]:
!pip install geoopt -q
import os, json, time, pickle
import numpy as np, torch, torch.nn as nn, geoopt
from collections import defaultdict, deque

ROOT_DIR=HME_ROOT
DATA=os.path.join(ROOT_DIR,'data/processed/icd10_tree_with_features.pkl')
CKPT_DIR=HME_ROOT + '/results/dim_runs'; os.makedirs(CKPT_DIR,exist_ok=True)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device',device)

with open(DATA,'rb') as f: data=pickle.load(f)
nodes,edges=data['nodes'],data['edges']
codes=list(nodes.keys()); code_to_idx={c:i for i,c in enumerate(codes)}; idx_to_code={i:c for c,i in code_to_idx.items()}
N=len(codes); edges_idx=[(code_to_idx[p],code_to_idx[c]) for p,c in edges]
ROOT=code_to_idx['ROOT']; ea=np.array(edges_idx)
connected=set(map(tuple,edges_idx))|set((v,u) for u,v in edges_idx)
print(f'N={N}, edges={len(edges_idx)}')

DIMS=[5,10,2]          # required order
ARMS=['pos','const']   # position-dependent always first

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device cuda
N=46817, edges=46816


## 2. Structure + derive c(A)

In [5]:
kids=defaultdict(list); parent_of={}
for u,v in edges_idx: kids[u].append(v); parent_of[v]=u
depth=np.full(N,-1,int); depth[ROOT]=0
bf=np.array([len(kids[i]) for i in range(N)],float)
A=np.zeros(N); q=deque([ROOT])
while q:
    x=q.popleft()
    for c_ in kids[x]:
        if depth[c_]<0: depth[c_]=depth[x]+1; A[c_]=A[x]+np.log1p(bf[x]); q.append(c_)
size=np.ones(N); hgt=np.zeros(N)
for i in sorted(range(N),key=lambda i:-depth[i]):
    for c_ in kids[i]: size[i]+=size[c_]; hgt[i]=max(hgt[i],hgt[c_]+1)
h_saf=np.maximum(hgt,1.0)

# c(u) proportional to ((log S + A)/h)^2, mapped to a usable curvature band
raw_c=((np.log(size)+A)/h_saf)**2
lo,hi=np.percentile(raw_c,[5,95])
C_MIN,C_MAX=0.25,1.0                      # was 0.5,2.0 — keeps 4x spread, ball radius = 1.0  # curvature band
c_node=np.clip((raw_c-lo)/(hi-lo+1e-12),0,1)*(C_MAX-C_MIN)+C_MIN
c_node[ROOT]=c_node[depth<=1].mean()

print(f'derived curvature c: min {c_node.min():.3f} | p25 {np.percentile(c_node,25):.3f} | '
      f'median {np.median(c_node):.3f} | p75 {np.percentile(c_node,75):.3f} | max {c_node.max():.3f}')
print(f'spread (max/min): {c_node.max()/c_node.min():.2f}x   <- needs to be >~1.5x to matter')
print('c by depth: '+'  '.join(f'd{d}:{c_node[depth==d].mean():.2f}' for d in range(int(depth.max())+1)))
np.save(f'{CKPT_DIR}/c_node.npy',c_node)

derived curvature c: min 0.250 | p25 0.583 | median 0.756 | p75 0.888 | max 1.000
spread (max/min): 4.00x   <- needs to be >~1.5x to matter
c by depth: d0:0.25  d1:0.25  d2:0.25  d3:0.35  d4:0.49  d5:0.73  d6:0.87  d7:0.87


## 3. DIAGNOSTIC — does varying c actually change the distance function?
If the same point pair gives nearly identical distance under c_min vs c_max, the whole idea is
dead before training. Check this before spending GPU time.

In [6]:
def d_c(u,v,c):
    num=np.sum((u-v)**2); den=(1-c*np.sum(u**2))*(1-c*np.sum(v**2))
    if den<=0: return float('nan')          # outside this curvature's ball
    return (1/np.sqrt(c))*np.arccosh(max(1+2*c*num/(den+1e-12),1.0))

CS=[float(c_node.min()), float(np.median(c_node)), float(c_node.max())]
print(f"testing actual band: c = {CS[0]:.2f}, {CS[1]:.2f}, {CS[2]:.2f}")
print(f"ball radius for each: {1/np.sqrt(CS[0]):.2f}, {1/np.sqrt(CS[1]):.2f}, {1/np.sqrt(CS[2]):.2f}")
print(f"\n{'|u|':>6}" + "".join(f"{'d(c='+format(c,'.2f')+')':>12}" for c in CS) + f"{'ratio':>8}")
for r in [0.3,0.5,0.7,0.85,0.95,0.99]:
    u=np.zeros(10); u[0]=r
    v=np.zeros(10); v[1]=r
    ds=[d_c(u,v,c) for c in CS]
    rat=ds[2]/ds[0] if (ds[0]==ds[0] and ds[2]==ds[2]) else float('nan')
    print(f"{r:>6.2f}" + "".join(f"{x:>12.3f}" for x in ds) + f"{rat:>8.2f}")
print("\nall rows should be finite (ball radius >= 1 for every c in band)")
print("ratio should rise monotonically with radius; >1.3 at r>=0.6 means c reshapes distances")

testing actual band: c = 0.25, 0.76, 1.00
ball radius for each: 2.00, 1.15, 1.00

   |u|   d(c=0.25)   d(c=0.76)   d(c=1.00)   ratio
  0.30       0.861       0.888       0.902    1.05
  0.50       1.475       1.609       1.681    1.14
  0.70       2.151       2.574       2.834    1.32
  0.85       2.720       3.615       4.344    1.60
  0.95       3.138       4.632       6.635    2.11
  0.99       3.316       5.181       9.894    2.98

all rows should be finite (ball radius >= 1 for every c in band)
ratio should rise monotonically with radius; >1.3 at r>=0.6 means c reshapes distances


## 4. Negatives + eval (MAP / mean rank / median rank / distortion, plus closure)

In [7]:
conn_codes=np.array(sorted({int(u)*N+int(v) for (u,v) in connected}),dtype=np.int64)
def sample_negs(anchors,K):
    Aa=np.asarray(anchors,dtype=np.int64); out=np.random.randint(0,N,(Aa.shape[0],K))
    for _ in range(6):
        bad=(out==Aa[:,None])|np.isin(Aa[:,None]*N+out,conn_codes); n=int(bad.sum())
        if n==0: break
        out[bad]=np.random.randint(0,N,n)
    return out

nbrs=defaultdict(set)
for u,v in edges_idx: nbrs[u].add(v); nbrs[v].add(u)
_rng=np.random.default_rng(42)
EVAL=[edges_idx[i] for i in _rng.choice(len(edges_idx),1000,replace=False)]
adj=defaultdict(list)
for u,v in edges_idx: adj[u].append(v); adj[v].append(u)
_r2=np.random.default_rng(0); DPAIRS=[]
for s_ in _r2.integers(0,N,2000):
    dd={int(s_):0}; qq=deque([int(s_)])
    while qq:
        x=qq.popleft()
        for y in adj[x]:
            if y not in dd: dd[y]=dd[x]+1; qq.append(y)
    t=int(_r2.integers(0,N))
    if t in dd and t!=s_: DPAIRS.append((int(s_),t,dd[t]))

def dist_all(pos,i,cvec=None):
    u=pos[i]
    c = 1.0 if cvec is None else float(cvec[i])     # query's curvature, single scalar
    d2=np.sum((pos-u)**2,axis=1)
    den=(1-c*np.sum(u**2))*(1-c*np.sum(pos**2,axis=1))
    return (1/np.sqrt(c))*np.arccosh(np.maximum(1+2*c*d2/(den+1e-12),1.0))

def eval3(pos,cvec=None):
    if np.isnan(pos).any(): return {k:float('nan') for k in ['MAP','mean_rank','median_rank','distortion']}
    ranks=[];aps=[]
    for (u,v) in EVAL:
        d=dist_all(pos,u,cvec); d[u]=np.inf; o=np.argsort(d)
        ranks.append(int(np.where(o==v)[0][0])+1)
        tr=nbrs[u]; h=0; pr=[]
        for j,nd in enumerate(o):
            if nd in tr: h+=1; pr.append(h/(j+1))
            if h==len(tr): break
        if pr: aps.append(np.mean(pr))
    ed=np.array([float(dist_all(pos,u,cvec)[v]) for (u,v,dg) in DPAIRS])
    gd=np.array([dg for (u,v,dg) in DPAIRS],float)
    cc=np.dot(ed,gd)/(np.dot(ed,ed)+1e-12); ranks=np.array(ranks)
    return {'MAP':float(np.mean(aps)),'mean_rank':float(np.mean(ranks)),
            'median_rank':float(np.median(ranks)),'distortion':float(np.mean(np.abs(cc*ed-gd)/gd))}

def ancestors(x):
    o=[];cur=x
    while cur in parent_of: cur=parent_of[cur]; o.append(cur)
    return o
depth_of={x:len(ancestors(x)) for x in range(N)}
_rt=np.random.default_rng(11); _cp=[]
for x in _rt.choice(N,4000,replace=False):
    for a_ in ancestors(int(x))[1:]: _cp.append((int(x),a_))
_rt.shuffle(_cp); CPNR=[(x,a_) for (x,a_) in _cp[:1000] if depth_of[a_]>0]
def eval_closure(pos,cvec=None):
    rr=[];rk=[]
    for (x,a_) in CPNR:
        d=dist_all(pos,x,cvec); d[x]=np.inf
        for w in nbrs[x]: d[w]=np.inf
        r=int(np.where(np.argsort(d)==a_)[0][0])+1; rk.append(r); rr.append(1/r)
    rk=np.array(rk)
    return {'closMRR':float(np.mean(rr)),'clos_med':float(np.median(rk)),'clos_h10':float(np.mean(rk<=10))}
print('eval ready')

eval ready


## 5. Trainer — per-node curvature in the metric
`arm='pos'` uses c_uv = sqrt(c_u*c_v) per pair. `arm='const'` uses c=1 everywhere.
Points are stored as raw coordinates and projected inside the ball of the *largest* curvature
each step, so no point escapes any region's boundary.

In [8]:
C_T=torch.tensor(c_node,dtype=torch.float32,device=device)

def dist_t(u,v,c_anchor):
    """c_anchor: curvature of the QUERY node, broadcast over all candidates."""
    d2=((u-v)**2).sum(-1)
    den=(1-c_anchor*(u*u).sum(-1))*(1-c_anchor*(v*v).sum(-1))
    x=1+2*c_anchor*d2/den.clamp_min(1e-9)
    return torch.acosh(x.clamp_min(1.0+1e-7))/torch.sqrt(c_anchor)

class PoincareEmb(nn.Module):
    def __init__(self,N,dim):
        super().__init__()
        self.m=geoopt.PoincareBall(c=1.0)          # optimizer geometry: unit ball
        self.emb=geoopt.ManifoldParameter(
            torch.empty(N,dim).uniform_(-1e-3,1e-3),manifold=self.m)

def train(tag, arm, dim, scale=1.0, epochs=1500, lr=50, K=50, seed=0, log=500):
    torch.manual_seed(seed); np.random.seed(seed)
    model=PoincareEmb(N,dim).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.emb.data[ROOT]=0.0
    E=np.array(edges_idx); best={'MAP':0}; bp=None
    cvec_np=c_node if arm=='pos' else None
    for ep in range(epochs):
        eff=lr*0.01 if ep<10 else lr                # burn-in
        for g in opt.param_groups: g['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]
            a=torch.tensor(b[:,0],device=device); p=torch.tensor(b[:,1],device=device)
            ng=torch.tensor(sample_negs(b[:,0],K),device=device)
            eu,ev,en=model.emb[a],model.emb[p],model.emb[ng]
            if arm=='pos':
                cp=C_T[a]                      # anchor's own curvature
                cn=C_T[a].unsqueeze(1)         # same anchor, broadcast over negatives
            else:
                cp=torch.ones_like(C_T[a]); cn=torch.ones_like(C_T[ng])
            dp=dist_t(eu,ev,cp)*scale
            dn=dist_t(eu.unsqueeze(1),en,cn)*scale
            rank=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            loss=rank+1.0*torch.relu(eu.norm(dim=-1)-ev.norm(dim=-1)+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.emb.grad is not None: model.emb.grad[ROOT]=0.0
            opt.step()
            with torch.no_grad(): model.emb.data[ROOT]=0.0
        if (ep+1)%log==0:
            pos=model.emb.detach().cpu().numpy()
            if np.isnan(pos).any():
                print(f'    NaN at ep {ep+1}',flush=True); break
            r=eval3(pos,cvec_np)
            if r['MAP']>best['MAP']: best=r; bp=pos.copy()
    torch.save({'emb':torch.tensor(bp),'metrics':best,'arm':arm,'dim':dim},f'{CKPT}/{tag}.pt')
    return best,bp,cvec_np
print('trainer ready (RiemannianSGD lr=50, per-node c in distance only)')

trainer ready (RiemannianSGD lr=50, per-node c in distance only)


## 6. Run — d = 5, 10, 2 ; position-dependent before constant ; resumable

In [13]:
import os
p=f'{CKPT}/posc_results.json'
os.remove(p) if os.path.exists(p) else print('no results file')
print('cleared — next run retrains everything')

cleared — next run retrains everything


In [ ]:
import json
r=json.load(open(RES))
r={k:v for k,v in r.items() if not k.startswith('pos_')}   # keep const, redo pos
json.dump(r,open(RES,'w'))
print(f'{len(r)} entries kept')

In [14]:
RES=f'{CKPT}/posc_results.json'
results=json.load(open(RES)) if os.path.exists(RES) else {}
SCALES=[1.0,2.23]; SEEDS=[0,1,2]

for dim in DIMS:                      # 5, 10, 2
    for arm in ARMS:                  # pos before const
        for s_ in SCALES:
            for seed in SEEDS:
                key=f'{arm}_d{dim}_x{s_}_s{seed}'
                if key in results:
                    print(f'  skip {key} (MAP {results[key]["MAP"]:.4f})',flush=True); continue
                t0=time.time(); print(f'  >> {key}',flush=True)
                best,pos,cv=train(key,arm,dim,scale=s_,seed=seed)
                cl=eval_closure(pos,cv)
                results[key]={**best,**cl}; json.dump(results,open(RES,'w'))
                print(f"  << {key}: MAP {best['MAP']:.4f} mean_rk {best['mean_rank']:.0f} "
                  f"med_rk {best['median_rank']:.0f} dist {best['distortion']:.4f} "
                  f"closMRR {cl['closMRR']:.4f} clos_med {cl['clos_med']:.0f} "
                  f"h@10 {cl['clos_h10']:.3f} ({(time.time()-t0)/60:.1f}m)",flush=True)

print(f"\n{'run':<20}{'MAP':>8}{'±std':>8}{'mean_rk':>9}{'med_rk':>8}{'distort':>9}{'closMRR':>9}")
for dim in DIMS:
    for arm in ARMS:
        for s_ in SCALES:
            got=[results[f'{arm}_d{dim}_x{s_}_s{sd}'] for sd in SEEDS
                 if f'{arm}_d{dim}_x{s_}_s{sd}' in results]
            if not got: continue
            g=lambda k: np.mean([x[k] for x in got])
            sd_=np.std([x['MAP'] for x in got])
            print(f"{arm+' d'+str(dim)+' x'+str(s_):<20}{g('MAP'):>8.4f}{sd_:>8.4f}"
              f"{g('mean_rank'):>9.0f}{g('median_rank'):>8.0f}{g('distortion'):>9.4f}"
              f"{g('closMRR'):>9.4f}{g('clos_med'):>10.0f}"+('' if len(got)==3 else f'  ({len(got)}/3)'))
    print()

  >> pos_d5_x1.0_s0
  << pos_d5_x1.0_s0: MAP 0.7082 mean_rk 1319 med_rk 3 dist 0.2668 closMRR 0.0251 clos_med 2527 h@10 0.085 (16.7m)
  >> pos_d5_x1.0_s1


KeyboardInterrupt: 

In [15]:
ck=torch.load(f'{CKPT}/pos_d5_x2.23_s1.pt',map_location='cpu'); pos=ck['emb'].numpy()
rng=np.random.default_rng(0)

# 1. triangle inequality violations
viol=[];gap=[]
for _ in range(3000):
    u,v,w=rng.integers(0,N,3)
    duv=dist_all(pos,u,c_node)[v]; dvw=dist_all(pos,v,c_node)[w]; duw=dist_all(pos,u,c_node)[w]
    if duw>duv+dvw+1e-6: viol.append(1); gap.append(duw-(duv+dvw))
    else: viol.append(0)
print(f"triangle violations: {100*np.mean(viol):.1f}% | mean excess {np.mean(gap) if gap else 0:.3f}")

# 2. who are the bad-rank nodes? stratify by curvature spread of the pair
ranks=[];cspread=[]
for (u,v) in EVAL[:400]:
    d=dist_all(pos,u,c_node); d[u]=np.inf
    ranks.append(int(np.where(np.argsort(d)==v)[0][0])+1)
    cspread.append(abs(c_node[u]-c_node[v]))
ranks=np.array(ranks); cspread=np.array(cspread)
hi=cspread>np.median(cspread)
print(f"pairs with LOW  curvature gap: median rank {np.median(ranks[~hi]):.0f}, mean {ranks[~hi].mean():.0f}")
print(f"pairs with HIGH curvature gap: median rank {np.median(ranks[hi]):.0f}, mean {ranks[hi].mean():.0f}")
print("  -> if HIGH-gap pairs are far worse, the mixed-curvature blend is the problem")

triangle violations: 0.2% | mean excess 0.556
pairs with LOW  curvature gap: median rank 3, mean 148
pairs with HIGH curvature gap: median rank 4, mean 481
  -> if HIGH-gap pairs are far worse, the mixed-curvature blend is the problem


In [16]:
r=json.load(open(RES)); r.pop('pos_d5_x1.0_s0',None); json.dump(r,open(RES,'w'))

In [21]:
# ============================================================
# BANDWIDTH SPREAD ON GO — self-contained. Reparses GO, loads saved checkpoint.
# ============================================================
import os, numpy as np, torch
from collections import defaultdict, deque

def load_emb(tag):
    ck=torch.load(f'{CKPT_DIR}/{tag}.pt',map_location='cpu')
    return ck['emb'].numpy(), ck.get('metrics',{})

def _dpoin(a, allp):
    d2=np.sum((allp-a)**2,axis=1); nu=1-np.sum(a**2); nv=1-np.sum(allp**2,axis=1)
    return np.arccosh(np.maximum(1+2*d2/(nu*nv+1e-12),1.0))

# ---- reparse GO (must match the parse used for go_const_x1.0: 38,245 terms) ----
if not os.path.exists('go-basic.obo'):
    !wget -q http://current.geneontology.org/ontology/go-basic.obo -O go-basic.obo
go_name={}; go_parents=defaultdict(list); cur=None; obs=set()
with open('go-basic.obo') as f:
    for line in f:
        line=line.strip()
        if line=='[Term]': cur={'id':None,'is_a':[],'obs':False}
        elif line=='[Typedef]': cur=None
        elif cur is not None:
            if line.startswith('id: GO:'): cur['id']=line[4:]
            elif line.startswith('name:') and cur['id']: go_name[cur['id']]=line[6:]
            elif line.startswith('is_obsolete: true'): cur['obs']=True
            elif line.startswith('is_a:'): cur['is_a'].append(line.split('!')[0].replace('is_a:','').strip())
            elif line=='':
                if cur['id'] and not cur['obs']: go_parents[cur['id']]=cur['is_a']
                if cur and cur['obs']: obs.add(cur['id'])
                cur=None
go_terms=[t for t in go_name if t not in obs]
go_to_idx={t:i for i,t in enumerate(go_terms)}; N_go=len(go_terms)
go_edges=[]
for ch in go_terms:
    for pa in go_parents[ch]:
        if pa in go_to_idx: go_edges.append((go_to_idx[pa],go_to_idx[ch]))
print(f'GO: {N_go} nodes, {len(go_edges)} edges')

pos_go,_=load_emb('go_const_x1.0')
assert pos_go.shape[0]==N_go, f"parse mismatch: checkpoint {pos_go.shape[0]} vs parse {N_go}"

# ---- depth = shortest path from any root (DAG, so BFS from all parentless nodes) ----
kids_go=defaultdict(list); has_parent=set()
for u,v in go_edges: kids_go[u].append(v); has_parent.add(v)
roots=[i for i in range(N_go) if i not in has_parent]
go_depth=np.full(N_go,-1,int); q=deque()
for r in roots: go_depth[r]=0; q.append(r)
while q:
    x=q.popleft()
    for c in kids_go[x]:
        if go_depth[c]<0: go_depth[c]=go_depth[x]+1; q.append(c)
go_depth[go_depth<0]=0
print(f'roots: {len(roots)} | max depth: {go_depth.max()}')

# ---- bandwidth measurement ----
rng=np.random.default_rng(0); ea_go=np.array(go_edges)
samp=rng.choice(len(ea_go),3000,replace=False)
by_d=defaultdict(list)
for i in samp:
    u,v=ea_go[i]
    d_all=_dpoin(pos_go[u],pos_go); d_all[u]=np.inf
    dn=d_all[rng.integers(0,N_go,500)]
    by_d[go_depth[v]].append(np.percentile(dn,5)-d_all[v])

print(f"\n{'depth':>6}{'n_edges':>9}{'hard m':>10}{'s*=1/m':>10}")
Tm={}
for d in sorted(by_d):
    if len(by_d[d])<30: continue
    m=np.median(by_d[d]); Tm[d]=m
    print(f"{d:>6}{len(by_d[d]):>9}{m:>10.3f}{1/max(m,1e-3):>10.3f}")

w=np.array([np.sum(go_depth[ea_go[:,1]]==d) for d in Tm])
bw=np.array([1/Tm[d] for d in Tm]); rep=np.repeat(bw,w)
print(f"\nGO raw spread:              {bw.max()/bw.min():.2f}x")
print(f"GO edge-weighted (p90/p10): {np.percentile(rep,90)/np.percentile(rep,10):.2f}x")
print(f"ICD-10 for comparison:      1.06x")
print("\n>1.5x -> criterion discriminates between graphs (positive control)")
print("~1.0x -> both uniform; criterion still untested on a heterogeneous graph")

GO: 38245 nodes, 57824 edges
roots: 3 | max depth: 13

 depth  n_edges    hard m    s*=1/m
     2       67    -2.348  1000.000
     3      291     0.140     7.146
     4      510     0.452     2.213
     5      849     0.507     1.974
     6      603     2.765     0.362
     7      428     3.439     0.291
     8      179     5.368     0.186
     9       40     6.703     0.149

GO raw spread:              -16.78x
GO edge-weighted (p90/p10): 7.61x
ICD-10 for comparison:      1.06x

>1.5x -> criterion discriminates between graphs (positive control)
~1.0x -> both uniform; criterion still untested on a heterogeneous graph


In [22]:
Tm_ok={d:m for d,m in Tm.items() if m>0}
print(f"excluded {len(Tm)-len(Tm_ok)} depth(s) with negative margin: "
      f"{[d for d,m in Tm.items() if m<=0]}")
w=np.array([np.sum(go_depth[ea_go[:,1]]==d) for d in Tm_ok])
bw=np.array([1/Tm_ok[d] for d in Tm_ok]); rep=np.repeat(bw,w)
print(f"GO raw spread:              {bw.max()/bw.min():.2f}x")
print(f"GO edge-weighted (p90/p10): {np.percentile(rep,90)/np.percentile(rep,10):.2f}x")
print(f"ICD-10:                     1.06x")
print(f"\nfraction of GO edges with negative hard margin at depth<=2: "
      f"{np.sum(go_depth[ea_go[:,1]]<=2)/len(ea_go):.3f}")

excluded 1 depth(s) with negative margin: [np.int64(2)]
GO raw spread:              47.90x
GO edge-weighted (p90/p10): 7.61x
ICD-10:                     1.06x

fraction of GO edges with negative hard margin at depth<=2: 0.024


In [23]:
# GO positional field: does adaptive curvature help where bandwidth spread is 7.61x?
import numpy as np
from collections import defaultdict, deque
ALPHA=-2.0

bf_go=np.array([len(kids_go[i]) for i in range(N_go)],float)
# cumulative ancestral log-branching, BFS from the 3 roots
A_go=np.zeros(N_go); seen=np.zeros(N_go,bool); q=deque()
for r in roots: seen[r]=True; q.append(r)
while q:
    x=q.popleft()
    for c in kids_go[x]:
        if not seen[c]: seen[c]=True; A_go[c]=A_go[x]+np.log1p(bf_go[x]); q.append(c)
d_saf_go=np.maximum(go_depth,1).astype(float)
INT_go=bf_go>0
print(f"GO internal nodes: {INT_go.sum()} ({100*INT_go.mean():.1f}%)  vs ICD-10's 23.0%")

raw=((np.log1p(bf_go)+A_go)/d_saf_go)**2
v=raw[INT_go]; a,b=np.percentile(v,[1,99])
k=np.full(N_go,np.nan); k[INT_go]=np.clip(2*(raw[INT_go]-a)/(b-a+1e-12)-1,-1,1)
parent_go={}
for u,v_ in go_edges: parent_go.setdefault(v_,u)
for x in np.argsort(go_depth):
    if np.isnan(k[x]) and x in parent_go: k[x]=k[parent_go[x]]
k_go=np.nan_to_num(k,nan=-1.0)

ea_g=np.array(go_edges)
me=np.exp(ALPHA*0.5*(k_go[ea_g[:,0]]+k_go[ea_g[:,1]]))
rng=np.random.default_rng(0); ra,rb=rng.integers(0,N_go,200000),rng.integers(0,N_go,200000)
mr=np.exp(ALPHA*0.5*(k_go[ra]+k_go[rb]))
print(f"GO field: std {k_go.std():.3f} | edge mean {me.mean():.3f} | logstd {np.std(np.log(me)):.3f}"
      f" | ASYM {mr.mean()/me.mean():.3f} | corr w/depth {np.corrcoef(k_go,go_depth)[0,1]:+.3f}")
print(f"  ICD-10 equivalent was: logstd 0.783, ASYM 0.984, corr w/depth -0.665")
np.save(f'{CKPT_DIR}/kappa_go.npy', k_go)

GO internal nodes: 13686 (35.8%)  vs ICD-10's 23.0%
GO field: std 0.527 | edge mean 1.665 | logstd 0.981 | ASYM 0.920 | corr w/depth -0.552
  ICD-10 equivalent was: logstd 0.783, ASYM 0.984, corr w/depth -0.665


In [24]:
# Flip to match the measured bandwidth pattern, then verify per-depth multipliers
# track the s* profile before spending GPU time.
k_go_f = -k_go * (k_go.std()/(-k_go).std())
np.save(f'{CKPT_DIR}/kappa_go_flip.npy', k_go_f)

for nm,k in [('original',k_go),('flipped',k_go_f)]:
    me=np.exp(ALPHA*0.5*(k[ea_g[:,0]]+k[ea_g[:,1]]))
    M=me.mean(); mr=np.exp(ALPHA*0.5*(k[ra]+k[rb]))
    print(f"\n{nm}: corr-depth {np.corrcoef(k,go_depth)[0,1]:+.3f} | logstd {np.std(np.log(me)):.3f}"
          f" | ASYM {mr.mean()/me.mean():.3f} | norm {M:.4f}")
    print(f"  {'depth':>6}{'s* wanted':>11}{'mult (norm)':>13}{'match?':>9}")
    for d in sorted(Tm_ok):
        m_ = me[go_depth[ea_g[:,1]]==d]
        if len(m_)<30: continue
        want=1/Tm_ok[d]; got=m_.mean()/M
        print(f"  {d:>6}{want:>11.2f}{got:>13.3f}{'  ok' if (want>1)==(got>1) else '  MISMATCH':>9}")

print("\nflipped should show mult>1 where s* is large (shallow) and mult<1 where s* is small (deep)")


original: corr-depth -0.552 | logstd 0.981 | ASYM 0.920 | norm 1.6647
   depth  s* wanted  mult (norm)   match?
       3       7.15        0.368  MISMATCH
       4       2.21        0.672  MISMATCH
       5       1.97        0.846  MISMATCH
       6       0.36        1.213  MISMATCH
       7       0.29        1.442  MISMATCH
       8       0.19        1.812  MISMATCH
       9       0.15        2.192  MISMATCH

flipped: corr-depth +0.552 | logstd 0.981 | ASYM 0.746 | norm 1.5175
   depth  s* wanted  mult (norm)   match?
       3       7.15        2.436       ok
       4       2.21        1.423       ok
       5       1.97        0.934  MISMATCH
       6       0.36        0.488       ok
       7       0.29        0.378       ok
       8       0.19        0.279       ok
       9       0.15        0.225       ok

flipped should show mult>1 where s* is large (shallow) and mult<1 where s* is small (deep)


In [25]:
# Can ASYM be neutralized without destroying the depth profile?
best=None
for shift in np.linspace(-0.3,0.3,13):
    k=np.clip(k_go_f+shift,-1,1)
    me=np.exp(ALPHA*0.5*(k[ea_g[:,0]]+k[ea_g[:,1]])); mr=np.exp(ALPHA*0.5*(k[ra]+k[rb]))
    asym=mr.mean()/me.mean(); rd=np.corrcoef(k,go_depth)[0,1]
    print(f"  shift {shift:+.2f}: ASYM {asym:.3f} | logstd {np.std(np.log(me)):.3f} | corr-depth {rd:+.3f}")
    if best is None or abs(asym-1.0)<abs(best[1]-1.0): best=(shift,asym,k)
print(f"\nbest shift {best[0]:+.2f} -> ASYM {best[1]:.3f}")
np.save(f'{CKPT_DIR}/kappa_go_flip_bal.npy', best[2])

  shift -0.30: ASYM 0.804 | logstd 0.876 | corr-depth +0.541
  shift -0.25: ASYM 0.795 | logstd 0.896 | corr-depth +0.543
  shift -0.20: ASYM 0.785 | logstd 0.914 | corr-depth +0.545
  shift -0.15: ASYM 0.776 | logstd 0.932 | corr-depth +0.547
  shift -0.10: ASYM 0.766 | logstd 0.948 | corr-depth +0.549
  shift -0.05: ASYM 0.756 | logstd 0.965 | corr-depth +0.551
  shift +0.00: ASYM 0.746 | logstd 0.981 | corr-depth +0.552
  shift +0.05: ASYM 0.746 | logstd 0.980 | corr-depth +0.552
  shift +0.10: ASYM 0.747 | logstd 0.978 | corr-depth +0.552
  shift +0.15: ASYM 0.747 | logstd 0.974 | corr-depth +0.552
  shift +0.20: ASYM 0.748 | logstd 0.969 | corr-depth +0.552
  shift +0.25: ASYM 0.750 | logstd 0.962 | corr-depth +0.551
  shift +0.30: ASYM 0.752 | logstd 0.953 | corr-depth +0.551

best shift -0.30 -> ASYM 0.804


In [26]:
# Three arms: const / flipped-positional / ASYM-matched control.
# The control has the SAME negative-deflation but NO depth structure,
# so a flipped win over control isolates position from the margin effect.
rng2=np.random.default_rng(7)
k_ctrl=k_go_f.copy(); rng2.shuffle(k_ctrl)
# rescale shuffled field so its ASYM matches flipped's
for sc in np.linspace(0.5,2.0,31):
    kt=np.clip(k_ctrl*sc,-1,1)
    me=np.exp(ALPHA*0.5*(kt[ea_g[:,0]]+kt[ea_g[:,1]])); mr=np.exp(ALPHA*0.5*(kt[ra]+kt[rb]))
    if abs(mr.mean()/me.mean()-0.746)<0.02: k_ctrl=kt; break
me=np.exp(ALPHA*0.5*(k_ctrl[ea_g[:,0]]+k_ctrl[ea_g[:,1]])); mr=np.exp(ALPHA*0.5*(k_ctrl[ra]+k_ctrl[rb]))
print(f"control: ASYM {mr.mean()/me.mean():.3f} | logstd {np.std(np.log(me)):.3f} "
      f"| corr-depth {np.corrcoef(k_ctrl,go_depth)[0,1]:+.3f}  (want ~0 depth corr)")
np.save(f'{CKPT_DIR}/kappa_go_ctrl.npy', k_ctrl)

control: ASYM 1.000 | logstd 0.741 | corr-depth -0.000  (want ~0 depth corr)


In [27]:
# Control = shuffled field (no depth structure) + explicit negative deflation
# matching flipped's ASYM. Now the ONLY difference from flipped is position.
ASYM_TARGET = 0.746
rng2=np.random.default_rng(7)
k_ctrl=k_go_f.copy(); rng2.shuffle(k_ctrl)
# match logstd to flipped's 0.981 by rescaling
me_f=np.exp(ALPHA*0.5*(k_go_f[ea_g[:,0]]+k_go_f[ea_g[:,1]]))
target_ls=np.std(np.log(me_f))
for sc in np.linspace(0.5,3.0,51):
    kt=np.clip(k_ctrl*sc,-1,1)
    me=np.exp(ALPHA*0.5*(kt[ea_g[:,0]]+kt[ea_g[:,1]]))
    if np.std(np.log(me))>=target_ls: k_ctrl=kt; break
me=np.exp(ALPHA*0.5*(k_ctrl[ea_g[:,0]]+k_ctrl[ea_g[:,1]]))
mr=np.exp(ALPHA*0.5*(k_ctrl[ra]+k_ctrl[rb]))
print(f"control: logstd {np.std(np.log(me)):.3f} (flipped {target_ls:.3f}) | "
      f"ASYM {mr.mean()/me.mean():.3f} | corr-depth {np.corrcoef(k_ctrl,go_depth)[0,1]:+.3f}")
print(f"-> apply ASYM_DEFLATE={ASYM_TARGET:.3f} to negatives in the control arm to match flipped")
np.save(f'{CKPT_DIR}/kappa_go_ctrl.npy', k_ctrl)

control: logstd 0.987 (flipped 0.981) | ASYM 0.995 | corr-depth -0.000
-> apply ASYM_DEFLATE=0.746 to negatives in the control arm to match flipped


In [28]:
# ============================================================
# GO: does adaptive curvature help where bandwidth spread is 7.61x?
# const / flipped (positional) / control (matched dispersion + ASYM, no position)
# ============================================================
import os, json, time
import numpy as np, torch, torch.nn as nn, geoopt
from collections import defaultdict

ALPHA=-2.0; ASYM_DEFLATE=0.746
k_flip=np.load(f'{CKPT_DIR}/kappa_go_flip.npy')
k_ctrl=np.load(f'{CKPT_DIR}/kappa_go_ctrl.npy')
KF=torch.tensor(k_flip,dtype=torch.float32,device=device)
KC=torch.tensor(k_ctrl,dtype=torch.float32,device=device)
M_F=float(np.exp(ALPHA*0.5*(k_flip[ea_g[:,0]]+k_flip[ea_g[:,1]])).mean())
M_C=float(np.exp(ALPHA*0.5*(k_ctrl[ea_g[:,0]]+k_ctrl[ea_g[:,1]])).mean())
print(f"normalizers: flipped {M_F:.4f} | control {M_C:.4f}")

conn_go=set(map(tuple,go_edges))|set((v,u) for u,v in go_edges)
cc_go=np.array(sorted({int(u)*N_go+int(v) for (u,v) in conn_go}),dtype=np.int64)
def negs_go(anchors,K):
    A_=np.asarray(anchors,dtype=np.int64); out=np.random.randint(0,N_go,(A_.shape[0],K))
    for _ in range(6):
        bad=(out==A_[:,None])|np.isin(A_[:,None]*N_go+out,cc_go); n=int(bad.sum())
        if n==0: break
        out[bad]=np.random.randint(0,N_go,n)
    return out

nb_go=defaultdict(set)
for u,v in go_edges: nb_go[u].add(v); nb_go[v].add(u)
_rg=np.random.default_rng(42)
EV_GO=[go_edges[i] for i in _rg.choice(len(go_edges),1000,replace=False)]
adj_go=defaultdict(list)
for u,v in go_edges: adj_go[u].append(v); adj_go[v].append(u)
from collections import deque as _dq
_r2=np.random.default_rng(0); DP_GO=[]
for s_ in _r2.integers(0,N_go,1500):
    dd={int(s_):0}; qq=_dq([int(s_)])
    while qq:
        x=qq.popleft()
        for y in adj_go[x]:
            if y not in dd: dd[y]=dd[x]+1; qq.append(y)
    t=int(_r2.integers(0,N_go))
    if t in dd and t!=s_: DP_GO.append((int(s_),t,dd[t]))

def eval_go(pos):
    if np.isnan(pos).any(): return {k:float('nan') for k in ['MAP','mean_rank','median_rank','distortion']}
    rk=[];aps=[]
    for (u,v) in EV_GO:
        d=_dpoin(pos[u],pos); d[u]=np.inf; o=np.argsort(d)
        rk.append(int(np.where(o==v)[0][0])+1)
        tr=nb_go[u]; h=0; pr=[]
        for j,nd in enumerate(o):
            if nd in tr: h+=1; pr.append(h/(j+1))
            if h==len(tr): break
        if pr: aps.append(np.mean(pr))
    ed=np.array([float(_dpoin(pos[u],pos[v:v+1])[0]) for (u,v,dg) in DP_GO])
    gd=np.array([dg for (u,v,dg) in DP_GO],float)
    c=np.dot(ed,gd)/(np.dot(ed,ed)+1e-12); rk=np.array(rk)
    return {'MAP':float(np.mean(aps)),'mean_rank':float(np.mean(rk)),
            'median_rank':float(np.median(rk)),'distortion':float(np.mean(np.abs(c*ed-gd)/gd))}

class PB(nn.Module):
    def __init__(s,n,d):
        super().__init__(); s.m=geoopt.PoincareBall(c=1.0)
        s.e=geoopt.ManifoldParameter(torch.empty(n,d).uniform_(-1e-3,1e-3),manifold=s.m)

def train_go(arm, scale=1.0, dim=10, epochs=1200, lr=50, K=50, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    model=PB(N_go,dim).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    E=np.array(go_edges); best={'MAP':0}; bp=None
    for ep in range(epochs):
        for g in opt.param_groups: g['lr']=lr*0.01 if ep<10 else lr
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]
            a=torch.tensor(b[:,0],device=device); p=torch.tensor(b[:,1],device=device)
            ng=torch.tensor(negs_go(b[:,1],K),device=device)
            eu,ev,en=model.e[a],model.e[p],model.e[ng]
            bp_=model.m.dist(eu,ev); bn=model.m.dist(eu.unsqueeze(1),en)
            if arm=='flip':
                bp_=bp_*torch.exp(ALPHA*0.5*(KF[a]+KF[p]))/M_F
                bn=bn*torch.exp(ALPHA*0.5*(KF[a].unsqueeze(1)+KF[ng]))/M_F
            elif arm=='ctrl':
                bp_=bp_*torch.exp(ALPHA*0.5*(KC[a]+KC[p]))/M_C
                bn=bn*torch.exp(ALPHA*0.5*(KC[a].unsqueeze(1)+KC[ng]))/M_C*ASYM_DEFLATE
            dp,dn=bp_*scale,bn*scale
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            loss=nk+1.0*torch.relu(ev.norm(dim=-1)-eu.norm(dim=-1)+0.05).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        if (ep+1)%400==0:
            pos=model.e.detach().cpu().numpy()
            if np.isnan(pos).any(): break
            r=eval_go(pos)
            if r['MAP']>best['MAP']: best=r; bp=pos.copy()
    return best,bp

RES=f'{CKPT_DIR}/go_pos_results.json'
res=json.load(open(RES)) if os.path.exists(RES) else {}
for arm in ['const','flip','ctrl']:
    for s_ in [1.0,2.23]:
        for seed in [0,1,2]:
            key=f'{arm}_x{s_}_s{seed}'
            if key in res: print(f'  skip {key}',flush=True); continue
            t0=time.time(); print(f'  >> {key}',flush=True)
            best,pos=train_go(arm,scale=s_,seed=seed)
            res[key]=best; json.dump(res,open(RES,'w'))
            torch.save({'emb':torch.tensor(pos)},f'{CKPT_DIR}/gp_{key}.pt')
            print(f"  << {key}: MAP {best['MAP']:.4f} mean_rk {best['mean_rank']:.0f} "
                  f"med_rk {best['median_rank']:.0f} dist {best['distortion']:.4f} "
                  f"({(time.time()-t0)/60:.1f}m)",flush=True)

print(f"\n{'arm':<8}{'s':>6}{'MAP':>9}{'±std':>8}{'mean_rk':>9}{'med_rk':>8}{'distort':>9}")
for arm in ['const','flip','ctrl']:
    for s_ in [1.0,2.23]:
        got=[res[f'{arm}_x{s_}_s{sd}'] for sd in [0,1,2] if f'{arm}_x{s_}_s{sd}' in res]
        if not got: continue
        g=lambda k: np.mean([x[k] for x in got])
        print(f"{arm:<8}{s_:>6}{g('MAP'):>9.4f}{np.std([x['MAP'] for x in got]):>8.4f}"
              f"{g('mean_rank'):>9.0f}{g('median_rank'):>8.0f}{g('distortion'):>9.4f}"
              +('' if len(got)==3 else f'  ({len(got)}/3)'))

normalizers: flipped 1.5175 | control 1.1764
  >> const_x1.0_s0
  << const_x1.0_s0: MAP 0.6138 mean_rk 144 med_rk 7 dist 0.2199 (19.0m)
  >> const_x1.0_s1
  << const_x1.0_s1: MAP 0.6133 mean_rk 115 med_rk 7 dist 0.2138 (19.2m)
  >> const_x1.0_s2
  << const_x1.0_s2: MAP 0.6084 mean_rk 131 med_rk 7 dist 0.2127 (19.0m)
  >> const_x2.23_s0
  << const_x2.23_s0: MAP 0.7287 mean_rk 33 med_rk 6 dist 0.2170 (19.0m)
  >> const_x2.23_s1
  << const_x2.23_s1: MAP 0.7344 mean_rk 25 med_rk 6 dist 0.2127 (19.0m)
  >> const_x2.23_s2
  << const_x2.23_s2: MAP 0.7327 mean_rk 31 med_rk 6 dist 0.2125 (18.9m)
  >> flip_x1.0_s0
  << flip_x1.0_s0: MAP 0.4562 mean_rk 1972 med_rk 21 dist 0.2184 (19.4m)
  >> flip_x1.0_s1
  << flip_x1.0_s1: MAP 0.4555 mean_rk 1935 med_rk 21 dist 0.2200 (19.4m)
  >> flip_x1.0_s2
  << flip_x1.0_s2: MAP 0.4565 mean_rk 1786 med_rk 21 dist 0.2147 (19.4m)
  >> flip_x2.23_s0
  << flip_x2.23_s0: MAP 0.4787 mean_rk 2314 med_rk 22 dist 0.2411 (19.4m)
  >> flip_x2.23_s1
  << flip_x2.23_s1: M

In [9]:
# ============================================================
# SARKAR CONSTRUCTION ON ICD-10 (2D Poincaré disk)
# Build in mpmath high precision -> store polar (r_hyp, theta) -> eval in float64
# ============================================================
!pip install mpmath -q
from mpmath import mp, mpf, mpc
import numpy as np, time
from collections import defaultdict, deque

kids=defaultdict(list); parent={}
for u,v in edges_idx: kids[u].append(v); parent[v]=u

def sarkar_2d(tau=1.0, dps=60):
    """Sarkar (2011) tree embedding. tau = hyperbolic edge length."""
    mp.dps=dps
    coord=[None]*N; coord[ROOT]=mpc(0,0)
    r_e=mp.tanh(mpf(tau)/2)                      # euclidean radius for hyp length tau
    k=len(kids[ROOT])
    for j,c in enumerate(kids[ROOT]):
        a=2*mp.pi*j/k
        coord[c]=r_e*mpc(mp.cos(a),mp.sin(a))
    q=deque(kids[ROOT])
    while q:
        v=q.popleft(); ch=kids[v]
        if not ch: continue
        x=coord[v]; xc=mp.conj(x)
        pl=(coord[parent[v]]-x)/(1-xc*coord[parent[v]])     # move v->0, see parent
        th_p=mp.atan2(pl.imag,pl.real)
        m=len(ch)
        for j,c in enumerate(ch):
            th=th_p+2*mp.pi*(j+1)/(m+1)                     # skip parent's slot
            loc=r_e*mpc(mp.cos(th),mp.sin(th))
            coord[c]=(loc+x)/(1+xc*loc)                     # move back
            q.append(c)
    r_h=np.zeros(N); th=np.zeros(N)
    for i in range(N):
        z=coord[i]; a=mp.sqrt(z.real**2+z.imag**2)
        r_h[i]=float(2*mp.atanh(a)) if a<1 else np.inf
        th[i]=float(mp.atan2(z.imag,z.real))
    return r_h,th

# ---- stable distance in polar form ----
# cosh(d) = cosh(dr) + 2*sinh(r_i)*sinh(r_j)*sin^2(dtheta/2)
def _logsinh(x):
    return np.where(x>20, x-np.log(2), np.log(np.sinh(np.clip(x,1e-300,None))))

def dist_from(i, r_h, th):
    d0=np.abs(th-th[i]); d0=np.minimum(d0,2*np.pi-d0)
    dr=np.abs(r_h-r_h[i])
    logB=(np.log(2)+_logsinh(np.full(N,r_h[i]))+_logsinh(r_h)
          +2*np.log(np.clip(np.sin(d0/2),1e-300,None)))
    far=logB>500
    out=np.empty(N)
    out[far]=np.log(2)+logB[far]
    ok=~far
    out[ok]=np.arccosh(np.maximum(np.cosh(np.clip(dr[ok],0,500))
                                  +np.exp(np.clip(logB[ok],-700,500)),1.0))
    return out

# ---- eval matching eval3: MAP / mean rank / median rank / distortion ----
nbrs=defaultdict(set)
for u,v in edges_idx: nbrs[u].add(v); nbrs[v].add(u)
_rg=np.random.default_rng(42)
EVAL=[edges_idx[i] for i in _rg.choice(len(edges_idx),1000,replace=False)]
adj=defaultdict(list)
for u,v in edges_idx: adj[u].append(v); adj[v].append(u)
_r2=np.random.default_rng(0); DP=[]
for s_ in _r2.integers(0,N,2000):
    dd={int(s_):0}; qq=deque([int(s_)])
    while qq:
        x=qq.popleft()
        for y in adj[x]:
            if y not in dd: dd[y]=dd[x]+1; qq.append(y)
    t=int(_r2.integers(0,N))
    if t in dd and t!=s_: DP.append((int(s_),t,dd[t]))

def eval_sarkar(r_h,th):
    if not np.isfinite(r_h).all():
        return {'MAP':float('nan'),'note':f'{(~np.isfinite(r_h)).sum()} non-finite'}
    ranks=[];aps=[]
    for (u,v) in EVAL:
        d=dist_from(u,r_h,th); d[u]=np.inf; o=np.argsort(d)
        ranks.append(int(np.where(o==v)[0][0])+1)
        tr=nbrs[u]; h=0; pr=[]
        for j,nd in enumerate(o):
            if nd in tr: h+=1; pr.append(h/(j+1))
            if h==len(tr): break
        if pr: aps.append(np.mean(pr))
    ed=np.array([dist_from(u,r_h,th)[v] for (u,v,_) in DP])
    gd=np.array([g for (_,_,g) in DP],float)
    c=np.dot(ed,gd)/(np.dot(ed,ed)+1e-12); ranks=np.array(ranks)
    return {'MAP':float(np.mean(aps)),'mean_rank':float(np.mean(ranks)),
            'median_rank':float(np.median(ranks)),
            'distortion':float(np.mean(np.abs(c*ed-gd)/gd))}

print(f"{'tau':>6}{'MAP':>9}{'mean_rk':>10}{'med_rk':>8}{'distort':>10}{'max_r':>8}{'time':>7}")
for tau in [0.5,1.0,2.0,3.0,5.0]:
    t0=time.time(); r_h,th=sarkar_2d(tau=tau)
    m=eval_sarkar(r_h,th)
    if 'note' in m: print(f"{tau:>6}  FAILED: {m['note']}"); continue
    print(f"{tau:>6}{m['MAP']:>9.4f}{m['mean_rank']:>10.0f}{m['median_rank']:>8.0f}"
          f"{m['distortion']:>10.4f}{r_h.max():>8.1f}{time.time()-t0:>7.0f}s")
    np.save(f'{CKPT_DIR}/sarkar_tau{tau}_r.npy',r_h)
    np.save(f'{CKPT_DIR}/sarkar_tau{tau}_th.npy',th)

   tau      MAP   mean_rk  med_rk   distort   max_r   time
   0.5   0.0020      2828    2978    0.3507     3.1     17s
   1.0   0.0052      1482    1400    0.2957     6.4     17s
   2.0   0.0808       163     100    0.1773    13.2     16s
   3.0   0.5781        13       7    0.1015    20.2     16s
   5.0   0.9997         4       3    0.0510    34.2     16s


In [16]:
# Self-contained: Sarkar 2D at tau=8, checking the plateau holds
!pip install mpmath -q
from mpmath import mp, mpf, mpc
import numpy as np, time
from collections import defaultdict, deque

kids=defaultdict(list); parent={}
for u,v in edges_idx: kids[u].append(v); parent[v]=u

def sarkar_2d(tau=1.0, dps=60):
    mp.dps=dps
    coord=[None]*N; coord[ROOT]=mpc(0,0)
    r_e=mp.tanh(mpf(tau)/2)
    k=len(kids[ROOT])
    for j,c in enumerate(kids[ROOT]):
        a=2*mp.pi*j/k
        coord[c]=r_e*mpc(mp.cos(a),mp.sin(a))
    q=deque(kids[ROOT])
    while q:
        v=q.popleft(); ch=kids[v]
        if not ch: continue
        x=coord[v]; xc=mp.conj(x)
        pl=(coord[parent[v]]-x)/(1-xc*coord[parent[v]])
        th_p=mp.atan2(pl.imag,pl.real)
        m=len(ch)
        for j,c in enumerate(ch):
            th=th_p+2*mp.pi*(j+1)/(m+1)
            loc=r_e*mpc(mp.cos(th),mp.sin(th))
            coord[c]=(loc+x)/(1+xc*loc)
            q.append(c)
    r_h=np.zeros(N); th=np.zeros(N)
    for i in range(N):
        z=coord[i]; a=mp.sqrt(z.real**2+z.imag**2)
        r_h[i]=float(2*mp.atanh(a)) if a<1 else np.inf
        th[i]=float(mp.atan2(z.imag,z.real))
    return r_h,th

def sarkar_auto(tau):
    dps=max(60,int(0.5*7*tau/np.log(10))+40)
    return sarkar_2d(tau=tau,dps=dps), dps

def _lsinh(x): return np.where(x>20,x-np.log(2),np.log(np.sinh(np.clip(x,1e-300,None))))
def dist_from(i,r_h,th):
    d0=np.abs(th-th[i]); d0=np.minimum(d0,2*np.pi-d0)
    dr=np.abs(r_h-r_h[i])
    lg=(np.log(2)+_lsinh(np.full(N,r_h[i]))+_lsinh(r_h)
        +2*np.log(np.clip(np.sin(d0/2),1e-300,None)))
    out=np.empty(N); far=lg>500
    out[far]=np.log(2)+lg[far]
    ok=~far
    out[ok]=np.arccosh(np.maximum(np.cosh(np.clip(dr[ok],0,500))+np.exp(np.clip(lg[ok],-700,500)),1.0))
    return out

nbrs=defaultdict(set)
for u,v in edges_idx: nbrs[u].add(v); nbrs[v].add(u)
_rg=np.random.default_rng(42)
EVAL=[edges_idx[i] for i in _rg.choice(len(edges_idx),1000,replace=False)]
adj=defaultdict(list)
for u,v in edges_idx: adj[u].append(v); adj[v].append(u)
_r2=np.random.default_rng(0); DP=[]
for s_ in _r2.integers(0,N,2000):
    dd={int(s_):0}; qq=deque([int(s_)])
    while qq:
        x=qq.popleft()
        for y in adj[x]:
            if y not in dd: dd[y]=dd[x]+1; qq.append(y)
    t=int(_r2.integers(0,N))
    if t in dd and t!=s_: DP.append((int(s_),t,dd[t]))

def eval_sarkar(r_h,th):
    if not np.isfinite(r_h).all():
        return {'MAP':float('nan'),'note':f'{(~np.isfinite(r_h)).sum()} non-finite'}
    ranks=[];aps=[]
    for (u,v) in EVAL:
        d=dist_from(u,r_h,th); d[u]=np.inf; o=np.argsort(d)
        ranks.append(int(np.where(o==v)[0][0])+1)
        tr=nbrs[u]; h=0; pr=[]
        for j,nd in enumerate(o):
            if nd in tr: h+=1; pr.append(h/(j+1))
            if h==len(tr): break
        if pr: aps.append(np.mean(pr))
    ed=np.array([dist_from(u,r_h,th)[v] for (u,v,_) in DP])
    gd=np.array([g for (_,_,g) in DP],float)
    c=np.dot(ed,gd)/(np.dot(ed,ed)+1e-12); ranks=np.array(ranks)
    return {'MAP':float(np.mean(aps)),'mean_rank':float(np.mean(ranks)),
            'median_rank':float(np.median(ranks)),
            'distortion':float(np.mean(np.abs(c*ed-gd)/gd))}

print(f"{'tau':>6}{'dps':>6}{'MAP':>9}{'mean_rk':>10}{'med_rk':>8}{'distort':>10}{'max_r':>8}{'time':>7}")
for tau in [5.0, 8.0, 12.0]:
    t0=time.time(); (r_h,th),dps=sarkar_auto(tau); m=eval_sarkar(r_h,th)
    if 'note' in m: print(f"{tau:>6}{dps:>6}  FAILED: {m['note']}"); continue
    print(f"{tau:>6}{dps:>6}{m['MAP']:>9.4f}{m['mean_rank']:>10.0f}{m['median_rank']:>8.0f}"
          f"{m['distortion']:>10.4f}{r_h.max():>8.0f}{time.time()-t0:>7.0f}s")
    np.save(f'{CKPT_DIR}/sarkar_tau{tau}_r.npy',r_h)
    np.save(f'{CKPT_DIR}/sarkar_tau{tau}_th.npy',th)

   tau   dps      MAP   mean_rk  med_rk   distort   max_r   time
   5.0    60   0.9997         4       3    0.0510      34     16s
   8.0    60   0.9988         4       3    0.0300      55     16s
  12.0    60   0.7019         9       5    0.0194      83     16s


In [19]:
# 2D-specific distance + eval, renamed so the nd version can't shadow them
def _lsinh(x): return np.where(x>20,x-np.log(2),np.log(np.sinh(np.clip(x,1e-300,None))))

def dist_from_2d(i, r_h, th):
    d0=np.abs(th-th[i]); d0=np.minimum(d0,2*np.pi-d0)
    dr=np.abs(r_h-r_h[i])
    lg=(np.log(2)+_lsinh(np.full(len(r_h),r_h[i]))+_lsinh(r_h)
        +2*np.log(np.clip(np.sin(d0/2),1e-300,None)))
    out=np.empty(len(r_h)); far=lg>500
    out[far]=np.log(2)+lg[far]
    ok=~far
    out[ok]=np.arccosh(np.maximum(np.cosh(np.clip(dr[ok],0,500))+np.exp(np.clip(lg[ok],-700,500)),1.0))
    return out

def eval_sarkar(r_h, th):
    if not np.isfinite(r_h).all():
        return {'MAP':float('nan'),'note':f'{(~np.isfinite(r_h)).sum()} non-finite'}
    ranks=[];aps=[]
    for (u,v) in EVAL:
        d=dist_from_2d(u,r_h,th); d[u]=np.inf; o=np.argsort(d)
        ranks.append(int(np.where(o==v)[0][0])+1)
        tr=nbrs[u]; h=0; pr=[]
        for j,nd in enumerate(o):
            if nd in tr: h+=1; pr.append(h/(j+1))
            if h==len(tr): break
        if pr: aps.append(np.mean(pr))
    ed=np.array([dist_from_2d(u,r_h,th)[v] for (u,v,_) in DP])
    gd=np.array([g for (_,_,g) in DP],float)
    c=np.dot(ed,gd)/(np.dot(ed,ed)+1e-12); ranks=np.array(ranks)
    return {'MAP':float(np.mean(aps)),'mean_rank':float(np.mean(ranks)),
            'median_rank':float(np.median(ranks)),
            'distortion':float(np.mean(np.abs(c*ed-gd)/gd))}

def sarkar_auto(tau):
    dps = int(7*tau/np.log(10)) + 50
    return sarkar_2d(tau=tau, dps=dps), dps

print(f"{'tau':>6}{'dps':>6}{'MAP':>9}{'mean_rk':>10}{'med_rk':>8}{'distort':>10}{'max_r':>8}{'time':>7}")
for tau in [8.0, 12.0, 16.0, 20.0]:
    t0=time.time(); (r_h,th),dps=sarkar_auto(tau); m=eval_sarkar(r_h,th)
    if 'note' in m: print(f"{tau:>6}{dps:>6}  FAILED: {m['note']}"); continue
    print(f"{tau:>6}{dps:>6}{m['MAP']:>9.4f}{m['mean_rank']:>10.0f}{m['median_rank']:>8.0f}"
          f"{m['distortion']:>10.4f}{r_h.max():>8.0f}{time.time()-t0:>7.0f}s")
    np.save(f'{CKPT_DIR}/sarkar_tau{tau}_r.npy',r_h)
    np.save(f'{CKPT_DIR}/sarkar_tau{tau}_th.npy',th)

   tau   dps      MAP   mean_rk  med_rk   distort   max_r   time
   8.0    74   0.9988         4       3    0.0300      55     17s
  12.0    86   0.7019         9       5    0.0194      83     18s
  16.0    98   0.3613        43      22    0.0148     111     18s
  20.0   110   0.1634       204      92    0.0192     139     19s


In [20]:
for tau in [5.0, 8.0, 12.0, 20.0]:
    r_h=np.load(f'{CKPT_DIR}/sarkar_tau{tau}_r.npy')
    th=np.load(f'{CKPT_DIR}/sarkar_tau{tau}_th.npy')
    uniq=len(np.unique(np.stack([r_h,th],1),axis=0))
    g=np.diff(np.sort(th)); g=g[g>0]
    print(f"tau={tau:>5}: unique coords {uniq}/{N} ({100*uniq/N:.1f}%) | "
          f"min θ gap {g.min():.2e} | float64 eps {np.spacing(1.0):.2e}")

tau=  5.0: unique coords 46817/46817 (100.0%) | min θ gap 2.22e-16 | float64 eps 2.22e-16
tau=  8.0: unique coords 46817/46817 (100.0%) | min θ gap 6.51e-19 | float64 eps 2.22e-16
tau= 12.0: unique coords 46657/46817 (99.7%) | min θ gap 3.18e-22 | float64 eps 2.22e-16
tau= 20.0: unique coords 43293/46817 (92.5%) | min θ gap 5.17e-26 | float64 eps 2.22e-16


In [17]:
# ============================================================
# SARKAR / DE SA CONSTRUCTION IN d DIMENSIONS + head-to-head vs const+temperature
# ============================================================
from mpmath import mp, mpf
import numpy as np, time
from collections import defaultdict, deque

kids=defaultdict(list); parent={}
for u,v in edges_idx: kids[u].append(v); parent[v]=u
MAXB=max(len(c) for c in kids.values())

# ---------- spherical codes: m+1 separated unit vectors, slot 0 = parent ----------
_code_cache={}
def spherical_code(m, dim, iters=1500, seed=0):
    key=(m,dim)
    if key in _code_cache: return _code_cache[key]
    n=m+1
    if n==1: X=np.eye(1,dim)
    else:
        rng=np.random.default_rng(seed)
        X=rng.normal(size=(n,dim)); X/=np.linalg.norm(X,axis=1,keepdims=True)
        lr=0.15
        for _ in range(iters):
            F=np.zeros_like(X)
            for i in range(n):
                diff=X[i]-X; dist=np.linalg.norm(diff,axis=1)+1e-9
                w=1.0/dist**3; w[i]=0
                F[i]=(diff*w[:,None]).sum(0)
            nrm=np.linalg.norm(F,axis=1,keepdims=True)+1e-12
            X+=lr*F/nrm; X/=np.linalg.norm(X,axis=1,keepdims=True); lr*=0.999
    _code_cache[key]=X
    return X

def min_angle(X):
    if len(X)<2: return np.pi
    D=X@X.T; np.fill_diagonal(D,-2.0)
    return float(np.arccos(np.clip(D.max(),-1,1)))

def rot_align(a,b):
    """rotation R with R@a=b, both unit"""
    c=float(np.dot(a,b)); d=len(a)
    if c>1-1e-12: return np.eye(d)
    if c<-1+1e-12:
        u=np.zeros(d); u[int(np.argmin(np.abs(a)))]=1.0
        u=u-np.dot(u,a)*a; u/=np.linalg.norm(u)
        return np.eye(d)-2*np.outer(u,u)
    u=b-c*a; u/=np.linalg.norm(u); s=np.sqrt(1-c*c)
    return np.eye(d)+(c-1)*(np.outer(a,a)+np.outer(u,u))+s*(np.outer(u,a)-np.outer(a,u))

print(f"max branching {MAXB} | threshold tau by dim (need d_sib > tau):")
for dim in [2,5,10]:
    ang=min_angle(spherical_code(MAXB,dim))
    print(f"  d={dim}: min angle {ang:.3f} rad -> tau > {-2*np.log(np.sin(ang/2)):.2f}")

# ---------- construction ----------
def mob_add(u,v,dim):
    uv=sum(u[i]*v[i] for i in range(dim)); u2=sum(x*x for x in u); v2=sum(x*x for x in v)
    a=1+2*uv+v2; b=1-u2; den=1+2*uv+u2*v2
    return [(a*u[i]+b*v[i])/den for i in range(dim)]

def sarkar_nd(dim, tau, dps=None):
    if dps is None: dps=max(60,int(0.5*7*tau/np.log(10))+40)
    mp.dps=dps
    r_e=mp.tanh(mpf(tau)/2)
    pos=[None]*N; pos[ROOT]=[mpf(0)]*dim
    q=deque()
    ch=kids[ROOT]
    if ch:
        C=spherical_code(len(ch)-1,dim)          # no parent slot at root
        for j,c in enumerate(ch):
            pos[c]=[r_e*mpf(float(C[j,k])) for k in range(dim)]
            q.append(c)
    while q:
        v=q.popleft(); ch=kids[v]
        if not ch: continue
        x=pos[v]; nx=[-t for t in x]
        pl=mob_add(nx,pos[parent[v]],dim)         # parent direction seen from v
        pn=float(mp.sqrt(sum(t*t for t in pl)))+1e-30
        pdir=np.array([float(t)/pn for t in pl])
        C=spherical_code(len(ch),dim)
        R=rot_align(C[0],pdir)                    # slot 0 -> parent
        CR=C@R.T
        for j,c in enumerate(ch):
            loc=[r_e*mpf(float(CR[j+1,k])) for k in range(dim)]
            pos[c]=mob_add(x,loc,dim); q.append(c)
    r_h=np.zeros(N); D=np.zeros((N,dim))
    for i in range(N):
        a=mp.sqrt(sum(t*t for t in pos[i]))
        r_h[i]=float(2*mp.atanh(a)) if a<1 else np.inf
        if a>0: D[i]=[float(t/a) for t in pos[i]]
        else: D[i,0]=1.0
    return r_h,D

# ---------- unified eval: everything converted to (r_hyp, unit dir) ----------
def to_polar(pos):
    n=np.linalg.norm(pos,axis=1); r=2*np.arctanh(np.clip(n,0,1-1e-15))
    D=pos/np.maximum(n,1e-30)[:,None]
    return r,D

def _lsinh(x): return np.where(x>20,x-np.log(2),np.log(np.sinh(np.clip(x,1e-300,None))))
def dist_from(i,r,D):
    ct=np.clip(D@D[i],-1,1); s2=np.clip((1-ct)/2,0,1)
    dr=np.abs(r-r[i])
    lg=np.log(2)+_lsinh(np.full(len(r),r[i]))+_lsinh(r)+np.log(np.clip(s2,1e-300,None))
    out=np.empty(len(r)); far=lg>500
    out[far]=np.log(2)+lg[far]
    ok=~far
    out[ok]=np.arccosh(np.maximum(np.cosh(np.clip(dr[ok],0,500))+np.exp(np.clip(lg[ok],-700,500)),1.0))
    return out

def eval_polar(r,D):
    if not np.isfinite(r).all(): return None
    ranks=[];aps=[]
    for (u,v) in EVAL:
        d=dist_from(u,r,D); d[u]=np.inf; o=np.argsort(d)
        ranks.append(int(np.where(o==v)[0][0])+1)
        tr=nbrs[u]; h=0; pr=[]
        for j,nd in enumerate(o):
            if nd in tr: h+=1; pr.append(h/(j+1))
            if h==len(tr): break
        if pr: aps.append(np.mean(pr))
    ed=np.array([dist_from(u,r,D)[v] for (u,v,_) in DP])
    gd=np.array([g for (_,_,g) in DP],float)
    c=np.dot(ed,gd)/(np.dot(ed,ed)+1e-12); ranks=np.array(ranks)
    return {'MAP':float(np.mean(aps)),'mean_rank':float(np.mean(ranks)),
            'median_rank':float(np.median(ranks)),
            'distortion':float(np.mean(np.abs(c*ed-gd)/gd))}

# ---------- run ----------
print(f"\n{'method':<26}{'d':>3}{'MAP':>9}{'mean_rk':>9}{'med_rk':>8}{'distort':>10}")
for dim in [2,5]:
    for tau in [3.0,5.0,8.0]:
        t0=time.time(); r,D=sarkar_nd(dim,tau); m=eval_polar(r,D)
        if m is None: print(f"{'sarkar tau='+str(tau):<26}{dim:>3}   non-finite"); continue
        print(f"{'sarkar tau='+str(tau):<26}{dim:>3}{m['MAP']:>9.4f}{m['mean_rank']:>9.0f}"
              f"{m['median_rank']:>8.0f}{m['distortion']:>10.4f}   ({time.time()-t0:.0f}s)")
        np.save(f'{CKPT_DIR}/sark_d{dim}_t{tau}_r.npy',r); np.save(f'{CKPT_DIR}/sark_d{dim}_t{tau}_D.npy',D)
    for s_ in ([1.0,2.23,5.0] if dim==2 else [1.0,2.23,5.0,8.0]):
        tag=f'msld_const_d{dim}_x{s_}_seed0'
        try: pos,_=load_emb(tag)
        except Exception: print(f"{'const x'+str(s_):<26}{dim:>3}   (missing {tag})"); continue
        r,D=to_polar(pos); m=eval_polar(r,D)
        print(f"{'const x'+str(s_):<26}{dim:>3}{m['MAP']:>9.4f}{m['mean_rank']:>9.0f}"
              f"{m['median_rank']:>8.0f}{m['distortion']:>10.4f}")

max branching 34 | threshold tau by dim (need d_sib > tau):
  d=2: min angle 0.163 rad -> tau > 5.02
  d=5: min angle 0.959 rad -> tau > 1.55
  d=10: min angle 1.275 rad -> tau > 1.04

method                      d      MAP  mean_rk  med_rk   distort
sarkar tau=3.0              2   0.5675       13       7    0.0994   (27s)
sarkar tau=5.0              2   0.9186        5       4    0.0495   (15s)
sarkar tau=8.0              2   0.3561       64      22    0.0297   (15s)
const x1.0                  2   (missing msld_const_d2_x1.0_seed0)
const x2.23                 2   (missing msld_const_d2_x2.23_seed0)
const x5.0                  2   (missing msld_const_d2_x5.0_seed0)
sarkar tau=3.0              5   1.0000        4       3    0.0267   (32s)
sarkar tau=5.0              5   0.9442        5       3    0.0147   (21s)
sarkar tau=8.0              5   0.3365       73      25    0.0093   (21s)
const x1.0                  5   (missing msld_const_d5_x1.0_seed0)
const x2.23                 5   (mis

In [22]:
# Does high-precision EVALUATION recover MAP at large tau?
from mpmath import mp, mpf
import numpy as np, time

def sarkar_2d_hp(tau, dps):
    """Same construction, but return mpmath r and theta (no float64 conversion)."""
    mp.dps=dps
    from mpmath import mpc
    coord=[None]*N; coord[ROOT]=mpc(0,0)
    r_e=mp.tanh(mpf(tau)/2); k=len(kids[ROOT])
    for j,c in enumerate(kids[ROOT]):
        a=2*mp.pi*j/k; coord[c]=r_e*mpc(mp.cos(a),mp.sin(a))
    q=deque(kids[ROOT])
    while q:
        v=q.popleft(); ch=kids[v]
        if not ch: continue
        x=coord[v]; xc=mp.conj(x)
        pl=(coord[parent[v]]-x)/(1-xc*coord[parent[v]])
        th_p=mp.atan2(pl.imag,pl.real); m=len(ch)
        for j,c in enumerate(ch):
            th=th_p+2*mp.pi*(j+1)/(m+1)
            loc=r_e*mpc(mp.cos(th),mp.sin(th))
            coord[c]=(loc+x)/(1+xc*loc); q.append(c)
    R=[];T=[]
    for i in range(N):
        z=coord[i]; a=mp.sqrt(z.real**2+z.imag**2)
        R.append(2*mp.atanh(a)); T.append(mp.atan2(z.imag,z.real))
    return R,T

def eval_hp(R,T,n_query=40):
    """MAP/rank via mpmath, ranking by cosh(d) (monotone in d)."""
    SH=[mp.sinh(r) for r in R]
    ranks=[];aps=[]
    for (u,v) in EVAL[:n_query]:
        ru,su,tu=R[u],SH[u],T[u]
        key=[]
        for j in range(N):
            if j==u: key.append(mp.inf); continue
            s=mp.sin((T[j]-tu)/2)
            key.append(mp.cosh(ru-R[j])+2*su*SH[j]*s*s)
        o=sorted(range(N),key=lambda j:key[j])
        ranks.append(o.index(v)+1)
        tr=nbrs[u]; h=0; pr=[]
        for jj,nd in enumerate(o):
            if nd in tr: h+=1; pr.append(h/(jj+1))
            if h==len(tr): break
        if pr: aps.append(float(np.mean(pr)))
    return float(np.mean(aps)), float(np.mean(ranks)), float(np.median(ranks))

print(f"{'tau':>6}{'dps':>6}{'MAP_f64':>10}{'MAP_hp':>9}{'mean_rk':>9}{'med_rk':>8}{'time':>7}")
f64={5.0:0.9997, 8.0:0.9988, 12.0:0.7019, 16.0:0.3613, 20.0:0.1634}
for tau in [8.0, 12.0, 20.0]:
    dps=int(7*tau/np.log(10))+50
    t0=time.time(); R,T=sarkar_2d_hp(tau,dps)
    mp.dps=dps
    a,mr,md=eval_hp(R,T,n_query=40)
    print(f"{tau:>6}{dps:>6}{f64[tau]:>10.4f}{a:>9.4f}{mr:>9.1f}{md:>8.0f}{time.time()-t0:>7.0f}s")
print("\nMAP_hp ~1.0 at tau=20 while MAP_f64=0.16 -> the loss is output precision, not the construction")

   tau   dps   MAP_f64   MAP_hp  mean_rk  med_rk   time
   8.0    74    0.9988   1.0000      3.6       3    109s
  12.0    86    0.7019   1.0000      3.6       3    118s
  20.0   110    0.1634   1.0000      3.9       3    142s

MAP_hp ~1.0 at tau=20 while MAP_f64=0.16 -> the loss is output precision, not the construction
